In [ ]:
# get raw data
import os
import librosa
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

MUSAN_PATH = r"D:\musan\musan"
SAMPLE_RATE = 16000
N_MFCC = 13
MAX_FILES_PER_CLASS = None
OUTPUT_DIR = "outputs_raw_data_plot"

def scan_musan(musan_path):
    records = []
    for label in ["speech", "music", "noise"]:
        class_path = os.path.join(musan_path, label)
        for root, _, files in os.walk(class_path):
            for f in files:
                if f.lower().endswith(".wav"):
                    records.append({
                        "path": os.path.join(root, f),
                        "label": label
                    })
    return pd.DataFrame(records)


def load_audio(path):
    y, sr = librosa.load(path, sr=SAMPLE_RATE, mono=True)
    return y, sr


def compute_duration(path):
    y, sr = load_audio(path)
    return librosa.get_duration(y=y, sr=sr)


def compute_rms(path):
    y, _ = load_audio(path)
    rms = librosa.feature.rms(y=y)
    return float(np.mean(rms))


def compute_zcr(path):
    y, _ = load_audio(path)
    zcr = librosa.feature.zero_crossing_rate(y)
    return float(np.mean(zcr))

# =========================
# MAIN
# =========================
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[1] Scanning MUSAN dataset...")
df = scan_musan(MUSAN_PATH)

if MAX_FILES_PER_CLASS:
    df = (
        df.groupby("label")
        .head(MAX_FILES_PER_CLASS)
        .reset_index(drop=True)
    )

print(df["label"].value_counts())

[1] Scanning MUSAN dataset...
label
noise     930
music     660
speech    426
Name: count, dtype: int64


In [5]:
# =========================
# DURATION
# =========================
df["duration"] = df["path"].apply(compute_duration)

plt.figure()
for label in df["label"].unique():
    plt.hist(
        df[df["label"] == label]["duration"],
        bins=50,
        alpha=0.5,
        label=label
    )
plt.legend()
plt.xlabel("Duration (seconds)")
plt.ylabel("Frequency")
plt.title("Duration Distribution")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/duration_dist.png", dpi=300)
plt.close()

KeyboardInterrupt: 

In [ ]:
# =========================
# RMS
# =========================
print("[3] Computing RMS energy...")
df["rms"] = df["path"].apply(compute_rms)

plt.figure()
for label in df["label"].unique():
    plt.hist(
        df[df["label"] == label]["rms"],
        bins=50,
        alpha=0.5,
        label=label
    )
plt.legend()
plt.xlabel("RMS Energy")
plt.ylabel("Count")
plt.title("RMS Energy Distribution")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/rms_dist.png", dpi=300)
plt.close()

In [ ]:
# =========================
# ZCR
# =========================
print("[4] Computing ZCR...")
df["zcr"] = df["path"].apply(compute_zcr)

plt.figure()
for label in df["label"].unique():
    plt.hist(
        df[df["label"] == label]["zcr"],
        bins=50,
        alpha=0.5,
        label=label
    )
plt.legend()
plt.xlabel("Zero Crossing Rate")
plt.ylabel("Count")
plt.title("ZCR Distribution")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/zcr_dist.png", dpi=300)
plt.close()


In [ ]:

    # =========================
    # PCA
    # =========================
    # print("[6] PCA visualization...")
    # X = mfcc_df.drop("label", axis=1).values
    # y = mfcc_df["label"].values

    # pca = PCA(n_components=2)
    # X_pca = pca.fit_transform(X)

    # plt.figure()
    # for label in np.unique(y):
    #     idx = y == label
    #     plt.scatter(
    #         X_pca[idx, 0],
    #         X_pca[idx, 1],
    #         label=label,
    #         alpha=0.5
    #     )
    # plt.legend()
    # plt.xlabel("PC1")
    # plt.ylabel("PC2")
    # plt.title("PCA on MFCC Features")
    # plt.tight_layout()
    # plt.savefig(f"{OUTPUT_DIR}/pca_mfcc.png", dpi=300)
    # plt.close()

    # # =========================
    # # SPECTROGRAM
    # # =========================
    # print("[7] Plotting spectrogram example...")
    # sample_path = df.iloc[0]["path"]
    # y, sr = load_audio(sample_path)

    # mel = librosa.feature.melspectrogram(y=y, sr=sr)
    # log_mel = librosa.power_to_db(mel)

    # plt.figure(figsize=(8, 4))
    # plt.imshow(log_mel, aspect="auto", origin="lower")
    # plt.colorbar()
    # plt.title("Log-Mel Spectrogram Example")
    # plt.xlabel("Time")
    # plt.ylabel("Mel bins")
    # plt.tight_layout()
    # plt.savefig(f"{OUTPUT_DIR}/logmel_example.png", dpi=300)
    # plt.close()

